In [1]:
!pip install datasets matplotlib pandas numpy seaborn scikit-learn tqdm

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 19.9 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 21.0 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 24.3 MB/s  0:00:00m0:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 15.7 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 26.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 16.6 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 563.4/563.4 kB 5.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 19.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 13.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 28.4 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 MB 38.9 MB/s  0:00:01m0:00:0100:01
   ━━━━━━━━━━━━━━

In [1]:
import pandas as pd
import numpy as np
from datasets import Dataset
import os
from tqdm import tqdm

from sklearn.metrics import confusion_matrix, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.utils import resample

import warnings
warnings.filterwarnings("ignore")

def evaluate_tfidf_classification(df, n_iterations=50, n_samples_per_class=308, max_features=10000):
    """
    Evaluate a TF-IDF based classifier on a balanced news classification task.

    Parameters
    ----------
    df : pd.DataFrame
        Must contain ['category_gold', 'text'] columns.
    n_iterations : int
        Number of resampling iterations.
    n_samples_per_class : int
        Number of samples per class for balancing.
    max_features : int
        Max number of features in TF-IDF.

    Returns
    -------
    results : dict
        Averaged precision, recall, f1, and accuracy per category.
    """

    metrics = {
        'Advertisement_precision': [], 'Advertisement_recall': [], 'Advertisement_f1': [],
        'National news_precision': [], 'National news_recall': [], 'National news_f1': [],
        'International news_precision': [], 'International news_recall': [], 'International news_f1': [],
        #'Paratext_precision': [], 'Paratext_recall': [], 'Paratext_f1': [],
        'accuracy': []
    }

    for i in tqdm(range(n_iterations), desc="TF-IDF classification"):
        # Balance by sampling equal number of items per class
        balanced_df = df.groupby('category_gold', group_keys=False).apply(
            lambda x: resample(x, n_samples=min(len(x), n_samples_per_class), random_state=i)
        ).sample(frac=1, random_state=i).reset_index(drop=True)

        # Labels
        y = balanced_df['category_gold'].values

        # TF-IDF features
        vectorizer = TfidfVectorizer(max_features=max_features)
        X = vectorizer.fit_transform(balanced_df['text'])

        # Train-test split
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=i, stratify=y
        )

        # Classifier
        clf = LogisticRegression(max_iter=1000, solver='lbfgs', multi_class='multinomial', random_state=i)
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)

        report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)

        for cat in ['Advertisement', 'National news', 'International news']: #, 'Paratext']:
            if cat in report:
                metrics[f'{cat}_precision'].append(report[cat]['precision'])
                metrics[f'{cat}_recall'].append(report[cat]['recall'])
                metrics[f'{cat}_f1'].append(report[cat]['f1-score'])
        metrics['accuracy'].append(report['accuracy'])

    # Average results
    averaged = {
        'model': 'TF-IDF',
        'Advertisement_precision': np.mean(metrics['Advertisement_precision']),
        'Advertisement_recall': np.mean(metrics['Advertisement_recall']),
        'Advertisement_f1': np.mean(metrics['Advertisement_f1']),
        'National news_precision': np.mean(metrics['National news_precision']),
        'National news_recall': np.mean(metrics['National news_recall']),
        'National news_f1': np.mean(metrics['National news_f1']),
        'International news_precision': np.mean(metrics['International news_precision']),
        'International news_recall': np.mean(metrics['International news_recall']),
        'International news_f1': np.mean(metrics['International news_f1']),
        #'Paratext_precision': np.mean(metrics['Paratext_precision']),
       # 'Paratext_recall': np.mean(metrics['Paratext_recall']),
       # 'Paratext_f1': np.mean(metrics['Paratext_f1']),
        'accuracy': np.mean(metrics['accuracy'])
    }

    return averaged


def evaluate_embeddings(df, embeddings, n_iterations=50, n_samples_per_class=308):
    """
    Evaluate multiple embedding models on a balanced news classification task.
    
    Parameters
    ----------
    df : pd.DataFrame
        Must contain ['category_gold', 'newspaper'] and columns with embeddings.
    embeddings : list[str]
        Column names of embedding vectors.
    n_iterations : int
        Number of resampling iterations per model.
    n_samples_per_class : int
        Number of samples per class for balancing.
    
    Returns
    -------
    results : list[dict]
        Average metrics per embedding model.
    """

    results = []

    # restrict to 3 categories
    #df = df[df['category_gold'].isin(['Advertisement', 'National news', 'International news'])]

    for embs in embeddings:
        metrics = {
            'model': embs,
            'Advertisement_precision': [], 'Advertisement_recall': [], 'Advertisement_f1': [],
            'National news_precision': [], 'National news_recall': [], 'National news_f1': [],
            'International news_precision': [], 'International news_recall': [], 'International news_f1': [],
            'Paratext_precision': [], 'Paratext_recall': [], 'Paratext_f1': [], # or without this
            'accuracy': []
        }

        for i in tqdm(range(n_iterations), desc=f"Testing {embs}"):
            # Balance dataset by sampling per class
            balanced_df = df.groupby('category_gold', group_keys=False).apply(
                lambda x: resample(x, n_samples=min(len(x), n_samples_per_class), random_state=i)
                ).sample(frac=1, random_state=i).reset_index(drop=True)

            # Features + labels
            X = np.vstack(balanced_df[embs].values)
            y = balanced_df['category_gold'].values

            # Stratify by category, not by newspaper
            X_train, X_test, y_train, y_test = train_test_split(
                X, y, test_size=0.2, random_state=i, stratify=y
            )

            clf = LogisticRegression(max_iter=1000, solver='lbfgs', multi_class='multinomial', random_state=i)
            clf.fit(X_train, y_train)
            y_pred = clf.predict(X_test)

            report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)

            for cat in ['Advertisement', 'National news', 'International news', 'Paratext']: # or without Paratext
                metrics[f'{cat}_precision'].append(report[cat]['precision'])
                metrics[f'{cat}_recall'].append(report[cat]['recall'])
                metrics[f'{cat}_f1'].append(report[cat]['f1-score'])
            metrics['accuracy'].append(report['accuracy'])

        # average results
        results.append({
            'model': embs,
            'Advertisement_precision': np.mean(metrics['Advertisement_precision']),
            'Advertisement_recall': np.mean(metrics['Advertisement_recall']),
            'Advertisement_f1': np.mean(metrics['Advertisement_f1']),
            'National news_precision': np.mean(metrics['National news_precision']),
            'National news_recall': np.mean(metrics['National news_recall']),
            'National news_f1': np.mean(metrics['National news_f1']),
            'International news_precision': np.mean(metrics['International news_precision']),
            'International news_recall': np.mean(metrics['International news_recall']),
            'International news_f1': np.mean(metrics['International news_f1']),
            'Paratext_precision': np.mean(metrics['Paratext_precision']), # or without these
            'Paratext_recall': np.mean(metrics['Paratext_recall']),
            'Paratext_f1': np.mean(metrics['Paratext_f1']),
            'accuracy': np.mean(metrics['accuracy'])
        })

    return results

In [2]:
# Load gold sample

sample_gold = pd.read_csv('../data/test_task/subset_final_gold_sample.csv', sep=',')
sample_gold.head()

FileNotFoundError: [Errno 2] No such file or directory: '../data/test_task/subset_final_gold_sample.csv'

In [4]:
sample_gold['newspaper'].unique()

array(['Aalborg Stiftstidende', 'Aalborgs Stifts Adresseavis',
       'Aarhuus Stifts-Tidende', 'Berlingske Tidende',
       'Den Nord-Cimbriske Tilskuer', 'Den Vest-Sjællandske Avis',
       'Efterretninger fra Adresse-Contoiret i Bergen',
       'Jyske Efterretninger', 'Københavns Adresseavis',
       'Lolland-Falsters Stifts-Tidende', 'Norske Intelligenssedler',
       'Odense Adresse-Contoirs Efterretninger',
       'Ribe Stifts Adresseaviser',
       'Tronhiems Adresse-Contoirs Efterretninger', 'Viborger Samler'],
      dtype=object)

In [5]:
# Load embeddings of sample

path_root = '../data/test_task/pooled/'

merged_all = Dataset.load_from_disk(f'{path_root}merged_all')
merged_all = merged_all.to_pandas()

In [5]:
# Merge gold labels with embeddings

embs_gold_orig = sample_gold.merge(merged_all[['article_id', 'e-5', 'jina', 'memo', 'old', 'bge', 'gemma']], on='article_id')
embs_gold_orig.shape

(1888, 12)

In [6]:
embs_gold_orig['category_gold'].value_counts()

category_gold
Advertisement         731
National news         511
International news    501
Paratext              145
Name: count, dtype: int64

In [13]:
# Baseline experiment

results_tfidf = evaluate_tfidf_classification(embs_gold_orig, n_iterations=50, n_samples_per_class=500)

TF-IDF classification: 100%|██████████| 50/50 [1:10:25<00:00, 84.51s/it]


In [ ]:
# Convert numpy floats to Python floats
results_clean = {k: float(v) if isinstance(v, np.generic) else v for k, v in results_tfidf.items()}

with open("../results/test_task/results_tfidf_3cats_250915.txt", "w") as f:
    for key, value in results_clean.items():
        f.write(f"{key}: {value}\n")

In [ ]:
# Example usage
embeddings = ['e-5', 'jina', 'memo', 'old', 'bge', 'gemma']
all_results = evaluate_embeddings(embs_gold_orig, embeddings, n_iterations=50, n_samples_per_class=500)

# Make a nice DataFrame
results_df = pd.DataFrame(all_results)

Testing e-5:   4%|▍         | 2/50 [00:05<02:12,  2.77s/it]


KeyboardInterrupt: 

In [19]:
results_df.to_csv('../results/test_task/results_4cats_250911.csv')

In [11]:
results_df.to_csv('../results/test_task/results_3cats_250911.csv')

In [10]:
embs_gold.groupby(['newspaper', 'category_gold'])['newspaper'].count()

NameError: name 'embs_gold' is not defined

In [55]:
# Take a sample of 100 from 11 newspapers, exclude the ones that are already in embs_gold
# Predict their category, save the silver sample
# Correct it and make it into a gold sample
# Add it to the current gold sample

newspapers = ['Aalborg Stiftstidende', 'Aalborgs Stifts Adresseavis',
       'Aarhuus Stifts-Tidende', 'Berlingske Tidende', 'Den Nord-Cimbriske Tilskuer',
       'Lolland-Falsters Stifts-Tidende', 'Odense Adresse-Contoirs Efterretninger',
       'Ribe Stifts Adresseaviser', 'Viborger Samler', 'Den Vest-Sjællandske Avis', 'Jyske Efterretninger']

# Get the set of article_ids you already used
used_ids = set(embs_gold_orig['article_id'])

# Filter out rows with those article_ids
extra_subset = merged_all[
    (merged_all['newspaper'].isin(newspapers)) &
    (~merged_all['article_id'].isin(used_ids))
]

# Sample up to 100 per newspaper
sample_extra = (
    extra_subset
    .groupby('newspaper', group_keys=False)
    .apply(lambda x: x.sample(n=min(len(x), 100), random_state=42))
    .reset_index(drop=True)
)

/tmp/ipykernel_506/3340524519.py:24: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=min(len(x), 100), random_state=42))


In [60]:
# Training data: labeled articles
X_train = np.vstack(embs_gold_orig['old'].values)
y_train = embs_gold_orig['category_gold'].values

# Train classifier
clf = LogisticRegression(max_iter=1000, solver='lbfgs', multi_class='multinomial', random_state=42)
clf.fit(X_train, y_train)

# Prepare test data: unseen sample
sample_extra = sample_extra[sample_extra['old_len'] == 768].copy()
X_test = np.vstack(sample_extra['old'].values)

# Predict categories
y_pred = clf.predict(X_test)

# Add predictions to DataFrame
sample_extra = sample_extra.copy()
sample_extra['category_predicted'] = y_pred

/home/ucloud/.local/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


In [61]:
sample_extra['category_predicted'].value_counts()

category_predicted
Advertisement         385
National news         357
International news    293
Paratext               61
Name: count, dtype: int64

In [62]:
sample_extra.columns

Index(['index', 'text', 'date', 'article_id', 'pwa', 'newspaper', 'e-5',
       'jina', 'memo', 'old', 'bge', 'gemma', 'old_len', 'category_predicted'],
      dtype='object')

In [63]:
sample_extra[['index', 'text', 'date', 'article_id', 'newspaper', 'category_predicted']].to_csv('../data/test_task/pooled/sample_extra_250911.csv')